# EDA - Swedavia FlightInfo (Sandbox data)

**Purpose:** Answer the questions about the dimensional model depending on: `Grain`, `Identity`, `Status`, `Delay`, `Codeshare`, `Date` with `measured` numbers.

**Source:** `data/sandbox_swedavia_merge.duckdb`, schema: `swedavia_raw`, loaded by `dlt` with `merge disposition` (Arrivals and departures).


In [3]:
# Connections cell
from pathlib import Path
import duckdb

# Notebook bor i notebooks/. En nivå upp är repo root
REPO_ROOT = Path.cwd().parent
DB_PATH = REPO_ROOT / "data" / "sandbox_swedavia_merge.duckdb"
assert DB_PATH.exists(), f"Did not find {DB_PATH}"

# read_only för att jag behöver bara läsa här än så länge, DuckDB tillåter endast EN writing process
con = duckdb.connect(str(DB_PATH), read_only=True)

## 1) The shape of the data

A) The tables size and timespan. How many rows, and what time window do the scheduled times cover?

B) Swedaviass airport field varies by direction:
- arrival -> arrival_airport, departure -> departure_airport
- AT TIME ZONE converts UTC to Swedish time BEFORE extracting the date

---

A)

| source_table | n_rows | first_scheduled | last_scheduled |
|:--|:--|:--|:--|
| arrivals| 2121 | 2026-10-05 00:00:00+02:00 | 2026-10-08 23:55:00+02:00 |
| departures | 2111 |2026-10-05 00:00:00+02:00 |2026-10-08 23:40:00+02:00 |

---

B) 

| direction	| swedavia_airport | 2026-10-05 | 2026-10-06 | 2026-10-07 | 2026-10-08 | 
|:--|:--|:--:|:--|:--|:--|
| arrival | ARN | 362 | 351 | 349 | 352 | 
| arrival | BMA | 20 | 23 | 23 | 22 | 
| arrival | GOT | 80 | 69 | 77 | 82 | 
| arrival | KRN | 2	| 2 | 2	| 2 | 
| arrival | LLA | 15 | 	13 | 15 | 15 | 
| arrival | MMX | 23 | 	21 | 22 | 23 | 
| arrival | OSD | 6 | 6 | 6 | 6 | 
| arrival | RNB | 6 | 6 | 6 | 6 | 
| arrival | UME | 15 | 	14 | 15 | 14 | 
| arrival | VBY | 13 | 	12 | 12 | 13 | 
| departure | ARN | 355 | 352 | 346 | 353 | 
| departure | BMA | 20 | 24 | 23 | 22 | 
| departure | GOT | 79 | 72 | 76 | 79 | 
| departure | KRN | 2 | 2 | 2 | 2 | 
| departure | LLA | 15 | 13 | 15 | 14 | 
| departure | MMX | 22 | 20 | 22 | 23 | 
| departure | OSD | 6 | 7 | 7 | 6 | 
| departure | RNB | 6 | 6 | 6 | 6 | 
| departure | UME | 14 | 14 | 16 | 14 | 
| departure | VBY | 13 | 12 | 12 | 13 | 



In [4]:
# Hur många rader och timespan räcker tiderna
con.sql("""
    SELECT 'arrivals' AS source_table,
            COUNT(*) AS n_rows,
            MIN(arrival_time__scheduled_utc) AS first_scheduled,
            MAX(arrival_time__scheduled_utc) AS last_scheduled
    FROM swedavia_raw.arrivals
    UNION ALL
    SELECT 'departures',
            COUNT(*),
            MIN(departure_time__scheduled_utc),
            MAX(departure_time__scheduled_utc)
    FROM swedavia_raw.departures
""").df()

,source_table,n_rows,first_scheduled,last_scheduled
0,arrivals,2121,2026-10-05 00:00:00+02:00,2026-10-08 23:55:00+02:00
1,departures,2111,2026-10-05 00:00:00+02:00,2026-10-08 23:40:00+02:00


In [5]:
# Swedavias flygplats står på olika sidor.
# PIVOT statement gör att varje datum blir en egen kolumn
con.sql("""
    WITH movements AS (
        SELECT 'arrival' AS direction,
               flight_leg_identifier__arrival_airport_iata AS swedavia_airport,
               CAST(arrival_time__scheduled_utc AT TIME ZONE 'Europe/Stockholm' AS DATE) AS local_date
        FROM swedavia_raw.arrivals
        UNION ALL
        SELECT 'departure',
               flight_leg_identifier__departure_airport_iata,
               CAST(departure_time__scheduled_utc AT TIME ZONE 'Europe/Stockholm' AS DATE)
        FROM swedavia_raw.departures
    )
    PIVOT movements ON local_date USING COUNT(*)
    GROUP BY direction, swedavia_airport
    ORDER BY direction, swedavia_airport
""").df()

,direction,swedavia_airport,2026-10-05,2026-10-06,2026-10-07,2026-10-08
0,arrival,ARN,362,351,349,352
1,arrival,BMA,20,23,23,22
2,arrival,GOT,80,69,77,82
3,arrival,KRN,2,2,2,2
4,arrival,LLA,15,13,15,15
5,arrival,MMX,23,21,22,23
6,arrival,OSD,6,6,6,6
7,arrival,RNB,6,6,6,6
8,arrival,UME,15,14,15,14
9,arrival,VBY,13,12,12,13


## 2) Identity. What IDENTIFIES one flight?

**Assumptions:** The flight leg identifier, i.e `flight_id` + `departure_date_utc` + `from_iata` + `to_iata` identifies exactly one flight per direction.

`legs` is used as a temporary `view` that should give both directions the same column names. Basically a preview of what the `dbt staging layer` will do later.

---

**Rows with UNIQUE flight identities:**
| direction | n_rows | n_flight_legs | n_duplicates | 
|:--|:--|:--|:--|
| arrival | 2121 | 2119 | 2 | 
| departure | 2111 | 2111 | 0 | 

---

**QUALIFY filtering:**
|direction |flight_id | departure_date_utc | from_iata | to_iata | status | scheduled_utc | actual_utc |
|:--|:--|:--|:--|:--|:--|:--|:--|
| arrival | 	BLX496	| 2026-10-07 02:00:00+02:00 |	RHO |	GOT |	LAN |	2026-10-08 00:50:00+02:00 |	2026-10-08 00:55:00+02:00 |
| arrival | 	BLX496	| 2026-10-07 02:00:00+02:00 |	RHO |	GOT | 	DEL |	2026-10-08 02:45:00+02:00 |	NaT |
| arrival |	D84528	| 2026-10-06 02:00:00+02:00 |	RAK |	ARN |	DEL |	2026-10-07 01:30:00+02:00 |	NaT |
| arrival |	D84528	| 2026-10-06 02:00:00+02:00 |	RAK |	ARN |	LAN | 	2026-10-07 02:10:00+02:00 |	2026-10-07 02:27:18+02:00 |

---

**Rows count but that Swedavia marked with DEL for deletion:**
| direction | n_rows | n_flight_legs | n_duplicates |
|:--|:--|:--|:--|
| arrival | 1769 | 1769 | 0 |
| departure | 1758 | 1758 | 0 |



In [6]:
# användandet av TEMP VIEW(view table), lever endast in memory, det skriver ingenting så read_only fungerar även här
# Samma column namn för båda riktningarna(arrival + departure) precis som staging layer i dbt ska göra.
con.sql("""
    CREATE OR REPLACE TEMP VIEW legs AS
    SELECT 'arrival' AS direction,
           flight_leg_identifier__flight_id AS flight_id,
           flight_leg_identifier__flight_departure_date_utc AS departure_date_utc,
           flight_leg_identifier__departure_airport_iata AS from_iata,
           flight_leg_identifier__arrival_airport_iata AS to_iata,
           location_and_status__flight_leg_status AS status,
           arrival_time__scheduled_utc AS scheduled_utc,
           arrival_time__actual_utc AS actual_utc
    FROM swedavia_raw.arrivals
    UNION ALL
    SELECT 'departure',
        flight_leg_identifier__flight_id,
        flight_leg_identifier__flight_departure_date_utc,
        flight_leg_identifier__departure_airport_iata,
        flight_leg_identifier__arrival_airport_iata,
        location_and_status__flight_leg_status,
        departure_time__scheduled_utc,
        departure_time__actual_utc
    FROM swedavia_raw.departures
""")

In [7]:
# Rader mot UNIKA flygidentiteter. Skillnaden är antalet duplicates
con.sql("""
    SELECT direction,
            COUNT(*) AS n_rows,
            COUNT(DISTINCT (flight_id, departure_date_utc, from_iata, to_iata)) AS n_flight_legs,
           n_rows - n_flight_legs AS n_duplicates
    FROM legs
    GROUP BY direction
    ORDER BY direction
""").df()

,direction,n_rows,n_flight_legs,n_duplicates
0,arrival,2121,2119,2
1,departure,2111,2111,0


In [8]:
# QUALIFY ska filtrera på en window function. Likt WHERE fast efter OVER()
# Ska finnas i Snowflake så mönstret ska fungera i dbt sen
# Koll för hur struligt det kommer bli med candidate keys

con.sql("""
    SELECT *
    FROM legs
    QUALIFY COUNT(*) OVER (
        PARTITION BY direction, flight_id, departure_date_utc, from_iata, to_iata
    ) > 1
    ORDER BY flight_id, scheduled_utc
""").df()


,direction,flight_id,departure_date_utc,from_iata,to_iata,status,scheduled_utc,actual_utc
0,arrival,BLX496,2026-10-07 02:00:00+02:00,RHO,GOT,LAN,2026-10-08 00:50:00+02:00,2026-10-08 00:55:00+02:00
1,arrival,BLX496,2026-10-07 02:00:00+02:00,RHO,GOT,DEL,2026-10-08 02:45:00+02:00,NaT
2,arrival,D84528,2026-10-06 02:00:00+02:00,RAK,ARN,DEL,2026-10-07 01:30:00+02:00,NaT
3,arrival,D84528,2026-10-06 02:00:00+02:00,RAK,ARN,LAN,2026-10-07 02:10:00+02:00,2026-10-07 02:27:18+02:00


In [9]:
# Samma räkning men utan rader som Swedavia marketat som DEL(borttagna)
con.sql("""
    SELECT direction,
           COUNT(*) AS n_rows,
           COUNT(DISTINCT (flight_id, departure_date_utc, from_iata, to_iata)) AS n_flight_legs,
           n_rows - n_flight_legs AS n_duplicates
    FROM legs
    WHERE status <> 'DEL'
    GROUP BY direction
    ORDER BY direction
""").df()

,direction,n_rows,n_flight_legs,n_duplicates
0,arrival,1769,1769,0
1,departure,1758,1758,0


### Findings related to question 2. identity

- With `DEL` rows: 
  - 2 duplicates were found, but only in `arrivals` table. 
- Without `DEL`: 
  - 0 duplicates were found in both directions.

- `DEL` = *Deleted* (Which points to a record removed by Swedavia) and *not* `CAN` = *Cancelled*. This is *two* different codes in the API spec.

- Both of the duplicates shows that Swedavia *deleted* the row and created a new one instead of updating it.


**What this means for the model:**
- This should indicate that the model will be notice when something has changed and will work if we were to add a `unique`-test on the `staging`-keys area after we're done filtering. IF **Swedavia** decides to change a time directly on an existing row one day instead of deleting it and creating a new one we would technically end up with *two* active rows sharing the *same* key.

- That would mean that the test would fail before the figure reaches the `serving layer dashboard`. Thinking in `W-A-P` terms, this would be the `A`(Audit) step. Audit and then Publish.

**The important parts for the modeling would then be these three parts:**

- 1) **Grain:** one row = one flight leg at a Swedavia airport, per direction.
- 2) **Key:** `direction` + `flight_id` + `departure_date_utc` + `from_iata` + `to_iata`, after removing `DEL`.
- 3) **Guard:** dbt `unique` test on the key in staging.

## 3) Which statuses count?

- Status distribution per direction(`arrivals`, `departures`) and how many of them actually has a time?


| direction | status | n_rows | percent | n_with_actual_time |
|:--|:--|:--|:--|:--|
| arrival | LAN | 1486 | 70.1 | 1486 |
| arrival | DEL | 352 | 16.6 | 0 |
| arrival | SCH | 267 | 12.6 | 0 |
| arrival | CAN | 16 | 0.8 | 0 |
| departure | ACT |  1497 |	70.9| 1497 |
| departure | DEL |  353 |	16.7| 1 |
| departure | SCH |  189 |	9.0	| 0 |
| departure | SEQ |  53 |	2.5	| 0 |
| departure | CAN |  19 |	0.9	| 1 |


In [10]:
# Fördelning på statusar per riktning (arrivals och departures). Hur många flyg har en riktig tid?

con.sql("""
    SELECT direction,
            status,
            COUNT(*) AS n_rows,
            ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (PARTITION BY direction), 1) AS percent,
            COUNT(actual_utc) AS n_with_actual_time
    FROM legs
    GROUP BY direction, status
    ORDER BY direction, n_rows DESC
""").df()

,direction,status,n_rows,percent,n_with_actual_time
0,arrival,LAN,1486,70.1,1486
1,arrival,DEL,352,16.6,0
2,arrival,SCH,267,12.6,0
3,arrival,CAN,16,0.8,0
4,departure,ACT,1497,70.9,1497
5,departure,DEL,353,16.7,1
6,departure,SCH,189,9.0,0
7,departure,SEQ,53,2.5,0
8,departure,CAN,19,0.9,1


In [11]:
# SCH status - Scheduled 
# SEQ status - Sequenced
# CAN status - Cancelled 
con.sql("""
    SELECT direction,
           status,
           CAST(scheduled_utc AT TIME ZONE 'Europe/Stockholm' AS DATE) AS local_date,
           COUNT(*) AS n_rows
    FROM legs
    WHERE status IN ('SCH', 'SEQ', 'CAN')
    GROUP BY ALL
    ORDER BY direction, status, local_date
""").df()

,direction,status,local_date,n_rows
0,arrival,CAN,2026-10-05,5
1,arrival,CAN,2026-10-06,2
2,arrival,CAN,2026-10-07,4
3,arrival,CAN,2026-10-08,5
4,arrival,SCH,2026-10-08,267
5,departure,CAN,2026-10-05,5
6,departure,CAN,2026-10-06,3
7,departure,CAN,2026-10-07,5
8,departure,CAN,2026-10-08,6
9,departure,SCH,2026-10-05,5
